# V-MD3 SAR Data Acquisition

Headless data collection for the V-MD3 radar SAR sweep experiment. This records one continuous right-to-left sweep of the Galil linear mover while the radar captures RADC / RFFT / DONE frames the whole time. Each sweep saves a `.bin` file with the raw frames and a `.json` file of the same name with the metadata (settings, stats, and the stage position of every RADC frame).

**Before running the notebook:** connect the radar and run the network setup (`run_setup.bat` file as administrator) so the radar responds at `192.168.100.201`. Close Galil Design Kit so it isn't holding the controller connection. Use the GUI tool (`python measure.py`) separately to view real-time heatmap. This notebook is for acquisition only.

**Workflow:**

- **Cell 1** - Imports and helper functions (run once).
- **Cell 2** - User configuration: session name, streams, sweep and stage settings (can edit these).
- **Cell 3** - Connect to the radar and the Galil, then home the stage to the right end (run once per session; re-run only if you change RSET, streams, or stage settings).
- **Cell 4** - Record one sweep (set the sweep name, then run cell; repeat per sweep).
- **Cell 5** - Disconnect (run at the end of the session).

The connections stay open between Cell 4 runs, and the stage returns to the right end on its own before each sweep. The Galil functions live in `lib/galil.py`, so restart the kernel after editing that file.

## Cell 1 - Imports and helper functions

In [ ]:
import json
import os
import socket
import time
from datetime import datetime

import numpy as np
import pandas as pd
from lib.galil import Stage
from lib.vmd3 import VMD3, RdotConfig, VMD3_SETTING

# Map user-facing stream names → RdotConfig flags and UDP headers
STREAM_MAP = {
    'radc': RdotConfig.RADC,
    'rfft': RdotConfig.RFFT,
    'done': RdotConfig.DONE,
}
HEADER_MAP = {
    'radc': b'RADC',
    'rfft': b'RFFT',
    'done': b'DONE',
}


def read_any_frame(vmd3, headers):
    """Read the next UDP frame whose header is in `headers`. Returns full frame."""
    while True:
        data = vmd3.recv_udp()
        if data[0:4] not in headers:
            continue
        resp_len = int.from_bytes(data[4:8], byteorder='little')
        while len(data) < resp_len + 8:
            data += vmd3.recv_udp()
        if resp_len != len(data[8:]):
            continue
        return data


def wait_for_frame(vmd3, headers):
    """Block until a full frame with one of `headers` arrives, riding through UDP timeouts."""
    while True:
        try:
            return read_any_frame(vmd3, headers)
        except socket.timeout:
            continue


def drain_udp(vmd3):
    """Throw away every frame already buffered so the next one read is fresh."""
    _old_timeout = vmd3.sockUDP.gettimeout()
    vmd3.sockUDP.settimeout(0.05)
    try:
        while True:
            try:
                vmd3.recv_udp()
            except socket.timeout:
                break
    finally:
        vmd3.sockUDP.settimeout(_old_timeout)


def write_metadata(json_path, bin_name, notes, aborted, started, ended, counts, sizes, frames, start_mm, end_mm):
    """Save the sweep settings, stats, and per-frame stage positions next to the .bin (settings from Cell 2)."""
    # Average RADC period actually measured over the sweep
    period = None
    if len(frames) > 1:
        period = (frames[-1]["t_s"] - frames[0]["t_s"]) / (len(frames) - 1)

    meta = {
        "file": bin_name,
        "notes": notes,
        "aborted": aborted,
        "start_time": started.isoformat(timespec="seconds"),
        "end_time": ended.isoformat(timespec="seconds"),
        "duration_s": round((ended - started).total_seconds(), 3),
        "radar": {
            "ip": RADAR_IP,
            "rset": RSET_CONFIG,
            "vmd3_setting": VMD3_SETTING[RSET_CONFIG],
            "streams": STREAMS,
            "frame_counts": counts,
            "frame_sizes": sizes,
            "target_frames": N_FRAMES,
            "frame_period_target_s": FRAME_PERIOD_S,
            "frame_period_measured_s": round(period, 4) if period else None,
        },
        "stage": {
            "galil_address": GALIL_ADDRESS,
            "axis": AXIS,
            "direction": "right_to_left",
            "counts_per_mm": COUNTS_PER_MM,
            "sweep_mm": SWEEP_MM,
            "speed_mm_s": round(SPEED_MM_S, 4),
            "accel_counts_s2": ACCEL,
            "ramp_mm": round(RAMP_MM, 3),
            "start_mm": round(start_mm, 4),
            "end_mm": round(end_mm, 4),
        },
        "frames": frames,
    }
    with open(json_path, "w") as f:
        json.dump(meta, f, indent=2)
    return period


print('[cell 1] Imports and helpers loaded.')

## Cell 2 - Configuration

Edit values below for the session, then run the cell. Re-run it any time something is changed.

- **Changing `SESSION_NAME` or the sweep settings** only needs this cell re-run — then go straight to Cell 4. No reconnect needed.
- **Changing `RSET_CONFIG`, `STREAMS`, or the Galil stage settings** needs a reconnect: re-run this cell, then Cell 3 (disconnects, reconnects, and re-homes).

`N_FRAMES` and `FRAME_PERIOD_S` set the stage speed so about that many RADC frames land across `SWEEP_MM` (101 frames over 100 mm comes out to ~7.63 mm/s). RADC is required since it clocks the sweep. RFFT and DONE, if selected, ride along with each RADC.

In [ ]:
# ─── Choose where to save binary data (e.g.) ───
OUTPUT_BASE   = "data"             # base directory for all sessions
SESSION_NAME  = "testdatafolder"   # folder name under OUTPUT_BASE, e.g. data/testdatafolder/

# ─── What to record ───
STREAMS = ["radc", "rfft", "done"]   # any of: "radc", "rfft", "done" (radc is required)

# ─── Sweep ───
SWEEP_MM = 100.0   # sweep length, right end to left end
N_FRAMES = 101   # RADC frames wanted across the sweep
FRAME_PERIOD_S = 0.131   # radar frame period, used to set the stage speed

# ─── Radar mode ───
RSET_CONFIG = 0   # 0 = 2D, 6 m max range, 4.69 cm range bin

# ─── If a sweep name was already recorded ───
ON_EXISTS = "suffix"   # "suffix" = save as _1, _2... | "overwrite" | "skip"

# ─── Galil stage ───
GALIL_ADDRESS = "192.168.0.100 --direct"   # same address GDK connects with
AXIS = "A"
RIGHT_IS_POSITIVE = True   # flip if homing drives to the left end instead
COUNTS_PER_MM = 10000   # CDS-3310, 1 count = 100 nm
ACCEL = 1000000   # counts/s^2, short ramps keep the sweep near constant speed
BACKOFF_MM = 1.0   # how far to back off the right end after homing
MOVE_SPEED_MM_S = 10.0   # speed for homing and returning to the right end

# ─── Connection (no need change) ───
RADAR_IP = "192.168.100.201"
UDP_TIMEOUT = 1.0              # seconds


# ─── Validate & normalize configurations (no need change) ───
STREAMS = [s.strip().lower() for s in STREAMS]
STREAMS = list(dict.fromkeys(STREAMS))           # remove duplicate entries, keep order

_bad = [s for s in STREAMS if s not in STREAM_MAP]
if _bad:
    raise ValueError(f"Unknown stream(s): {_bad}. Valid options: radc, rfft, done")
if "radc" not in STREAMS:
    raise ValueError("STREAMS must include radc, since RADC frames clock the sweep")
if N_FRAMES < 2:
    raise ValueError("N_FRAMES must be at least 2")
if not (0 <= RSET_CONFIG <= 8):
    raise ValueError("RSET_CONFIG must be 0-8")
if ON_EXISTS not in ("suffix", "overwrite", "skip"):
    raise ValueError('ON_EXISTS must be "suffix", "overwrite", or "skip"')
if SWEEP_MM + BACKOFF_MM > 102.0:
    raise ValueError("SWEEP_MM + BACKOFF_MM is longer than the ~102 mm rail")

SESSION_DIR = os.path.join(OUTPUT_BASE, SESSION_NAME)
BINS_DIR    = os.path.join(SESSION_DIR, "bins")

# Compute RDOT flag list from the selected streams
SELECTED_RDOT = [STREAM_MAP[s] for s in STREAMS]

# Stage speed so N_FRAMES land across the sweep
SWEEP_TIME_S = (N_FRAMES - 1) * FRAME_PERIOD_S
SPEED_MM_S = SWEEP_MM / SWEEP_TIME_S
RAMP_MM = (SPEED_MM_S * COUNTS_PER_MM) ** 2 / (2 * ACCEL) / COUNTS_PER_MM

print(f"[cell 2] Session folder : {SESSION_DIR}")
print(f"[cell 2] Streams        : {STREAMS}  (RDOT = 0x{sum(r.value for r in SELECTED_RDOT):02X})")
print(f"[cell 2] Sweep          : {SWEEP_MM:.1f} mm, {N_FRAMES} frames over {SWEEP_TIME_S:.2f} s")
print(f"[cell 2] Stage speed    : {SPEED_MM_S:.3f} mm/s (ramp {RAMP_MM:.2f} mm each end)")
print(f"[cell 2] RSET mode      : {RSET_CONFIG}")
print(f"[cell 2] On existing    : {ON_EXISTS}")

## Cell 3 - Connect to the radar and the Galil

Run once at the start of a session. This opens the radar connection, configures the radar with the RSET mode and selected streams, and creates the session folder. It then connects to the Galil and homes the stage to the right end (drives into the end, backs off `BACKOFF_MM`, and zeros there). The radar connects first so a dead radar fails before anything moves.

Safe to re-run: if a connection is already open, it closes it first. Re-run this after changing `RSET_CONFIG`, `STREAMS`, or the stage settings in Cell 2.

The connections stay open until you run Cell 5 (or restart the kernel), so you **don't** need to re-run this between Cell 4 sweeps.

In [ ]:
# IF a connection is already open from a previous run, close it first so re-running this cell doesn't leave a dangling socket.
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 3] Closed existing connection.")
    except Exception as e:
        print(f"[cell 3] Warning closing old connection: {e}")
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
    vmd3 = None

# Same for the Galil
if 'stage' in globals() and stage is not None:
    stage.close()
    print("[cell 3] Closed existing Galil connection.")
    stage = None

# ─── Create the session folders (committed here, not in Cell 2) ───
os.makedirs(BINS_DIR, exist_ok=True)

# ─── Connect ────────────────────────────────────────────────────
print(f"[cell 3] Connecting to V-MD3 at {RADAR_IP}...")
vmd3 = VMD3(tcp_ip=RADAR_IP)
vmd3.connect()
vmd3.set_rset_config(RSET_CONFIG)
vmd3.set_output_config(SELECTED_RDOT)
vmd3.sockUDP.settimeout(UDP_TIMEOUT)

print(f"[cell 3] Radar ready. RSET={RSET_CONFIG}, streams={STREAMS}, "
      f"RDOT=0x{sum(r.value for r in SELECTED_RDOT):02X}")

# ─── Connect to the Galil and home to the right end ───
print(f"[cell 3] Connecting to Galil at {GALIL_ADDRESS}...")
stage = Stage(GALIL_ADDRESS, AXIS, RIGHT_IS_POSITIVE, COUNTS_PER_MM, ACCEL)
print(f"[cell 3] {stage.g.GInfo()}")
print("[cell 3] Homing to the right end...")
stage.home_right(BACKOFF_MM, MOVE_SPEED_MM_S)

print("[cell 3] Ready. Stage homed at the right end (0 mm).")

## Cell 4 - Record one sweep

Set `SWEEP_NAME`, then run the cell. Repeat for each sweep.

The stage returns to the right end if it isn't there already, then the sweep starts the moment a fresh RADC frame arrives (that RADC is frame 0). Every selected stream is saved to one `.bin` file (`<SWEEP_NAME>.bin`), each RADC is tagged with the stage position, and recording stops on the first RADC after the stage stops. The metadata is saved next to it as `<SWEEP_NAME>.json`.

Interrupting the cell (stop button) stops the stage and still saves the `.json`, marked `"aborted": true`.

In [ ]:
SWEEP_NAME = "sweep"   # ← saves bins/sweep.bin + bins/sweep.json, then Shift+Enter
NOTES = ""   # optional note saved in the .json

def record_sweep(sweep_name, notes=""):
    """Record one right-to-left sweep and save its .bin and .json."""
    accepted = tuple(HEADER_MAP[s] for s in STREAMS)

    # ─── Build the output path, depends on ON_EXISTS ───
    base_name = sweep_name
    filepath = os.path.join(BINS_DIR, base_name + ".bin")

    if os.path.exists(filepath):
        if ON_EXISTS == "skip":
            print(f"[cell 4] {base_name}.bin already exists — SKIPPED "
                  f"(ON_EXISTS='skip').")
            return
        elif ON_EXISTS == "overwrite":
            print(f"[cell 4] {base_name}.bin exists — overwriting.")
        elif ON_EXISTS == "suffix":
            i = 1
            while os.path.exists(os.path.join(BINS_DIR, f"{base_name}_{i}.bin")):
                i += 1
            filepath = os.path.join(BINS_DIR, f"{base_name}_{i}.bin")
            print(f"[cell 4] {base_name}.bin exists — saving as "
                  f"{os.path.basename(filepath)}.")

    filename = os.path.basename(filepath)
    json_path = os.path.splitext(filepath)[0] + ".json"

    # ─── Record ───
    per_stream = {s: 0 for s in STREAMS}   # how many of each we have
    sizes = {s: 0 for s in STREAMS}
    header_to_name = {HEADER_MAP[s]: s for s in STREAMS}
    frames = []   # time and stage position of each RADC, for the .json
    n_written = 0
    bytes_written = 0
    aborted = False

    # ─── Park at the right end and preload the sweep so BG is the only command at the trigger ───
    stage.move_to_zero(MOVE_SPEED_MM_S)
    stage.preload_sweep(SWEEP_MM, SPEED_MM_S)
    start_mm = stage.state()[0]

    start_time = datetime.now().astimezone()

    print(f"[cell 4] Sweeping {SWEEP_MM:.1f} mm at {SPEED_MM_S:.3f} mm/s → {filename}  "
          f"(streams: {STREAMS})")

    out = open(filepath, "wb")
    try:
        # Align on a fresh RADC, start the stage on it, and keep it as frame 0
        drain_udp(vmd3)
        frame = wait_for_frame(vmd3, (b'RADC',))
        stage.start()
        t0 = time.perf_counter()
        t, x = 0.0, start_mm

        # Write every frame and tag each RADC with the stage position until the stage stops
        while True:
            stream = header_to_name[frame[0:4]]
            out.write(frame)
            per_stream[stream] += 1
            sizes[stream] = len(frame)
            n_written += 1
            bytes_written += len(frame)
            if stream == "radc":
                frames.append({"frame": len(frames), "t_s": round(t, 4), "x_mm": round(x, 4)})
                print(f"   RADC   frame {len(frames) - 1:3d}   t = {t:6.2f} s   x = {x:7.3f} mm")

            frame = wait_for_frame(vmd3, accepted)
            if frame[0:4] == b'RADC':
                t = time.perf_counter() - t0
                x, moving = stage.state()
                if not moving:
                    break   # this RADC arrived after the stage stopped, so it isn't saved
    except KeyboardInterrupt:
        stage.stop()
        aborted = True
        print("[cell 4] Interrupted, stage stopped. Saving what was recorded.")
    finally:
        out.close()

    stop_time = datetime.now().astimezone()
    duration = (stop_time - start_time).total_seconds()
    stage.wait()
    end_mm = stage.state()[0]

    # ─── Save the metadata next to the .bin ───
    period = write_metadata(json_path, filename, notes, aborted, start_time, stop_time,
                            per_stream, sizes, frames, start_mm, end_mm)

    print(f"[cell 4] Saved {filename} + {os.path.basename(json_path)}: {n_written} frames "
          f"({per_stream['radc']} RADC, target {N_FRAMES}), {bytes_written/1e6:.2f} MB, {duration:.1f} s")
    if period:
        print(f"[cell 4] Stage ended at {end_mm:.2f} mm, measured frame period "
              f"{period * 1000:.1f} ms (target {FRAME_PERIOD_S * 1000:.1f} ms)")

# ─── Run it for the sweep name set above ───
record_sweep(SWEEP_NAME, NOTES)

## Cell 5 - Disconnect

Run at the end of a session to close the radar and Galil connections cleanly.

Safe to run even if nothing is connected since it just reports that there's nothing to close.

In [ ]:
if 'vmd3' in globals() and vmd3 is not None:
    try:
        vmd3.sockTCP.settimeout(2.0)
        vmd3.disconnect()
        print("[cell 5] Disconnected from V-MD3.")
    except Exception as e:
        print(f"[cell 5] Disconnect warning: {e}")
        # Force-close the sockets so the OS releases the ports
        try:
            vmd3.sockTCP.close()
        except Exception:
            pass
        try:
            vmd3.sockUDP.close()
        except Exception:
            pass
        print("[cell 5] Sockets force-closed.")
    finally:
        vmd3 = None
else:
    print("[cell 5] Nothing to disconnect (no open connection).")

# ─── Close the Galil ───
if 'stage' in globals() and stage is not None:
    stage.close()
    print("[cell 5] Closed Galil connection.")
    stage = None
else:
    print("[cell 5] No Galil connection to close.")

## Cell 6 - Convert .bin to long-format CSV

Decodes a saved `.bin` file into a lossless "long" CSV: one row per (frame, bin, chirp, channel) value, with complex I/Q split into separate `I` and `Q` columns.

Set `BIN_TO_CONVERT` to the file you want, choose which streams to export, then run.

Set `OUTPUT_CSV_PATH` to the directory and file to save csv data.

Warning: long format is large - one RADC or RFFT frame is 32,768 rows, so a 5-frame-each file is ~330,000 rows. Lossless but big.

In [ ]:
# ─── What to convert ───
BIN_TO_CONVERT = os.path.join(BINS_DIR, "scan_0cm.bin")   # path to a .bin file
CONVERT_STREAMS = ["radc", "rfft"]     # which streams to export: radc and/or rfft
CONVERT_MODE = "2D"                    # "2D" or "3D" (matches how it was recorded)

# ─── Where to save ───
OUTPUT_CSV_PATH = "C:/My Computer/vmd3/test/data/csv/scan_0cm.csv"


# ─── Decoders ───
def _get_frames(filepath, header, expected_len):
    """Extract all valid payloads for a given header from a .bin file."""
    with open(filepath, "rb") as f:
        raw = f.read()
    frames, start = [], 0
    while True:
        idx = raw.find(header, start)
        if idx == -1:
            break
        if idx + 8 > len(raw):
            break
        payload_len = int.from_bytes(raw[idx+4:idx+8], byteorder="little")
        if payload_len == expected_len:
            p0, p1 = idx + 8, idx + 8 + expected_len
            if p1 <= len(raw):
                payload = raw[p0:p1]
                if header not in payload:   # reject header-in-payload false hits
                    frames.append(payload)
        start = idx + 1
    return frames


def _decode_radc_2d(frame):
    raw = np.frombuffer(frame, dtype="<i2").reshape(64, 4, 256)
    q = raw[:, :, 0::2].astype(np.float64)
    i = raw[:, :, 1::2].astype(np.float64)
    complex_data = i + 1j * q                       # (chirp=64, ch=4, samples=128)
    return np.transpose(complex_data, (2, 0, 1))    # (samples, chirps, channels)


def _decode_rfft_2d(frame):
    raw = np.frombuffer(frame, dtype="<i2").reshape(4, 128, 64, 2)
    i = raw[:, :, :, 1].astype(np.float64)
    q = raw[:, :, :, 0].astype(np.float64)
    complex_data = i + 1j * q                      # (ch, range_bin, chirp)
    return np.transpose(complex_data, (1, 2, 0))   # (range_bin, chirp, ch)


# ─── Build the long dataframe ───
def bin_to_long_df(filepath, streams, mode):
    expected_len = 131072 if mode == "2D" else 196608
    header_map = {"radc": b"RADC", "rfft": b"RFFT"}
    all_rows = []

    for s in streams:
        header = header_map[s]
        frames = _get_frames(filepath, header, expected_len)
        if not frames:
            print(f"  [{s}] no frames found — skipping")
            continue

        for f_idx, frame in enumerate(frames):
            if s == "radc":
                cube = _decode_radc_2d(frame)
            else:   # rfft 
                cube = _decode_rfft_2d(frame)

            # cube shape: (bin, chirp, channel) — flatten to index arrays
            n_bin, n_chirp, n_chan = cube.shape
            bb, cc, hh = np.meshgrid(
                np.arange(n_bin), np.arange(n_chirp), np.arange(n_chan),
                indexing="ij"
            )
            df = pd.DataFrame({
                "file":    os.path.basename(filepath),
                "stream":  s,
                "frame":   f_idx,
                "bin":     bb.ravel(),   # sample index (radc) or range bin (rfft)
                "chirp":   cc.ravel(),
                "channel": hh.ravel(),
                "I":       np.real(cube).ravel(),
                "Q":       np.imag(cube).ravel(),
            })
            all_rows.append(df)
            print(f"  [{s}] frame {f_idx}: {len(df)} rows")

    if not all_rows:
        print("No data converted.")
        return None
    return pd.concat(all_rows, ignore_index=True)


# ─── Run ────────────────────────────────────────────────────────
print(f"[cell 6] Converting {os.path.basename(BIN_TO_CONVERT)} "
      f"(streams={CONVERT_STREAMS}, mode={CONVERT_MODE})...")

df = bin_to_long_df(BIN_TO_CONVERT, CONVERT_STREAMS, CONVERT_MODE)

if df is not None:
    # Decide output path: manual override, or auto next to the .bin
    if OUTPUT_CSV_PATH.strip():
        out_csv = OUTPUT_CSV_PATH.strip()
        out_dir = os.path.dirname(out_csv)
        if out_dir:
            os.makedirs(out_dir, exist_ok=True)
    else:
        out_csv = os.path.splitext(BIN_TO_CONVERT)[0] + "_long.csv"

    df.to_csv(out_csv, index=False)
    print(f"[cell 6] Wrote {out_csv}")
    print(f"[cell 6] Shape: {df.shape[0]} rows × {df.shape[1]} cols")

    # ─── Per-stream row breakdown ───────────────────────────────
    stream_counts = df["stream"].value_counts()
    for s in CONVERT_STREAMS:
        n = int(stream_counts.get(s, 0))
        print(f"[cell 6]   {s}: {n} rows")
    print(f"[cell 6]   total: {len(df)} rows")

    print(df.head())